# Chapter 12 · Foundation Models for Materials — Colab notebook

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch12-foundation/>

We load MACE-MP-0, a pre-trained universal interatomic potential, and use it zero-shot: no training, no fine-tuning. We evaluate energies and forces on a crystal it has never explicitly seen, then run a short molecular-dynamics trajectory — all from a downloaded checkpoint.

This notebook is meant for **Google Colab** rather than the in-browser JupyterLite kernel, because it needs heavy packages (and, where noted, a GPU) that cannot run under Pyodide. Open it in Colab, and where the install cell mentions it, switch the runtime to a GPU via **Runtime -> Change runtime type -> GPU** before running the rest.


## Install

`mace-torch` provides both the MACE library and the `mace_mp` foundation-model loader, which downloads the pre-trained MACE-MP-0 checkpoint on first use. `torch` comes pre-installed on Colab. **Set the runtime to GPU** (Runtime -> Change runtime type -> GPU) for the molecular-dynamics cell; the single-point evaluation is fine on CPU.

In [ ]:
!pip install mace-torch


## Load the MACE-MP-0 foundation model

`mace_mp` is the foundation-model loader. The first call downloads the checkpoint (the `medium` model is a good default) and returns an ASE calculator ready to attach to any structure — no training step.

In [ ]:
import torch
from mace.calculators import mace_mp

device = 'cuda' if torch.cuda.is_available() else 'cpu'
calc = mace_mp(model='medium', dispersion=False,
               default_dtype='float64', device=device)
print('MACE-MP-0 loaded on', device)


## Build a structure and run a zero-shot evaluation

We construct an MgO rock-salt supercell and attach the foundation model. The energy and forces come straight out — zero-shot — with no system-specific training. This is the headline capability of a materials foundation model.

In [ ]:
import numpy as np
from ase.build import bulk

mgo = bulk('MgO', crystalstructure='rocksalt', a=4.21).repeat((3, 3, 3))
mgo.calc = calc

energy = mgo.get_potential_energy()
forces = mgo.get_forces()
print(f'{len(mgo)} atoms')
print(f'total energy   = {energy:.4f} eV')
print(f'energy / atom  = {energy / len(mgo):.4f} eV')
print(f'max |force|    = {np.abs(forces).max():.4e} eV/A  '
      '(near zero: the ideal lattice is close to equilibrium)')


## Probe a rattled structure

Displace the atoms slightly and the forces become non-trivial — the foundation model returns a restoring force pushing every atom back towards its lattice site, exactly as a DFT calculation would.

In [ ]:
rng = np.random.default_rng(0)
rattled = mgo.copy()
rattled.rattle(stdev=0.05, rng=rng)
rattled.calc = calc
print(f'rattled energy    = {rattled.get_potential_energy():.4f} eV')
print(f'rattled max|force| = {np.abs(rattled.get_forces()).max():.4f} eV/A')


## A short zero-shot molecular-dynamics run

Because the foundation model returns forces, we can drive molecular dynamics with it directly. Here is a brief NVT Langevin trajectory at 600 K — short enough to finish quickly, long enough to show the temperature settling around its target. Use the GPU runtime for this cell.

In [ ]:
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.units import fs

md_atoms = mgo.copy()
md_atoms.calc = calc
MaxwellBoltzmannDistribution(md_atoms, temperature_K=600.0)
dyn = Langevin(md_atoms, timestep=1.0 * fs, temperature_K=600.0,
               friction=0.01)

energies, temperatures = [], []
def record():
    energies.append(float(md_atoms.get_potential_energy()))
    temperatures.append(float(md_atoms.get_temperature()))
dyn.attach(record, interval=1)
dyn.run(100)
print(f'final energy = {energies[-1]:.3f} eV')
print(f'mean T over last 50 steps = {np.mean(temperatures[-50:]):.1f} K')


## Plot the trajectory

Energy and temperature versus step. The thermostat draws the temperature towards 600 K while the potential energy responds — the expected signature of an equilibrating NVT run.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(energies)
ax1.set_xlabel('MD step')
ax1.set_ylabel('potential energy (eV)')
ax1.set_title('Energy')
ax2.plot(temperatures)
ax2.axhline(600.0, ls='--', color='grey', label='target 600 K')
ax2.set_xlabel('MD step')
ax2.set_ylabel('temperature (K)')
ax2.set_title('Temperature')
ax2.legend()
fig.tight_layout()
plt.show()
